# Simple RAG - Query and Retrieval

**Flow:**

Query → Retrieve → LLM → Answer

## Install Dependencies

In [12]:
# # Conda environment setup
# !pip install langchain langchain-chroma langchain-openai chromadb

## Basic RAG Code Phase 2 - Query and Retrieval

In [13]:
# # Colab setup
# from google.colab import drive
# from google.colab import userdata
# drive.mount('/content/drive')

In [14]:
# # Colab Store Location
# store_location = "/content/drive/MyDrive/rag_langchain/data/chroma_db1"

In [15]:
# # Colab Key
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [16]:
# Local setup
store_location = "vector_db/chroma_db1"

In [17]:
# Use Python dotenv to load environment variables from a .env file
from dotenv import load_dotenv
import os

load_dotenv()

True

### Step 1: Create retriever

In [18]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=store_location,
    embedding_function=embedding
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Step 2: Build Modern RAG Chain (LCEL)

In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant.
Answer the question based only on the context below.

Context:
{context}

Question:
{question}
""")

### Step 3: Compose the chain

In [20]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Step 4: Query

In [21]:
response = rag_chain.invoke("What is this document about?")
print(response)

The document appears to discuss a past issue related to data exposure that occurred around 2013 and 2014, which affected many people. It mentions that the industry took steps to fix this problem. The context suggests that personal information, such as names, email addresses, and possibly phone numbers, was involved, likely in relation to a contest or promotion where individuals had a chance to win something.


## Debug

In [22]:
docs = retriever.invoke("What is this document about?")

for i, d in enumerate(docs):
    print(f'chunk {i}: {d.page_content}')
    print("-" * 50)

chunk 0: was fixed back in 2013 and 2014, 12 years ago.  At the time, many people were exposed.  We fixed that.  We as an industry fixed it.
--------------------------------------------------
chunk 1: was fixed back in 2013 and 2014, 12 years ago.  At the time, many people were exposed.  We fixed that.  We as an industry fixed it.
--------------------------------------------------
chunk 2: site, at least their names and email addresses and probably more.  Sometimes their phone numbers.  Because, hey, there's a chance to win.
--------------------------------------------------
chunk 3: site, at least their names and email addresses and probably more.  Sometimes their phone numbers.  Because, hey, there's a chance to win.
--------------------------------------------------
